# Baseline: Greedy Tote Sequencing

Greedy baseline that chooses the next tote with the lowest immediate transition cost from current tote/bin state.

This is a fast heuristic baseline against the exact model.

In [3]:
import csv
from pathlib import Path

# Choose which generated input run(s) to use.
# - RUN_ID = None  -> canonical inputs/
# - RUN_ID = int   -> one run folder inputs/runs/run_XXXX
# - RUN_ID = "all" -> all run folders under inputs/runs/
RUN_ID = "all"


def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]


def _get_input_bases(run_id):
    if run_id == "all":
        runs_root = _resolve_existing([Path("inputs/runs"), Path("../inputs/runs")])
        if not runs_root.exists():
            return []
        return sorted([p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")])
    if run_id is None:
        return [_resolve_existing([Path("inputs"), Path("../inputs")])]
    run_name = f"run_{int(run_id):04d}"
    return [_resolve_existing([Path("inputs/runs") / run_name, Path("../inputs/runs") / run_name])]


output_base = _resolve_existing([Path("outputs"), Path("../outputs")])
output_base.mkdir(parents=True, exist_ok=True)
OUT = output_base

NUM_CONVEYORS = 4
PLACE_TIME = 1.75
TOTE_SWITCH_TIME = 4.0
BIN_SWITCH_TIME = 0.75

# Composite objective = sum_order_completion_times - OPTIONALITY_LAMBDA * optionality_score
OPTIONALITY_LAMBDA = 0.35
OPT_EDGE_WEIGHT = 1.0
OPT_BRANCH_WEIGHT = 0.6
OPT_RARE_WEIGHT = 0.4


def _coerce_int(v):
    s = v.strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except ValueError:
        return None


def _read_rows(p):
    with p.open("r", newline="") as f:
        return list(csv.reader(f))


def build_blocks():
    ir = _read_rows(INPUT_ITEMTYPES)
    qr = _read_rows(INPUT_QUANTITIES)
    tr = _read_rows(INPUT_TOTES)
    n = max(len(ir), len(qr), len(tr))

    tote_bins = {}
    tote_items = {}
    for i in range(n):
        row_i = ir[i] if i < len(ir) else []
        row_q = qr[i] if i < len(qr) else []
        row_t = tr[i] if i < len(tr) else []
        w = max(len(row_i), len(row_q), len(row_t))
        for j in range(w):
            item = _coerce_int(row_i[j]) if j < len(row_i) else None
            qty = _coerce_int(row_q[j]) if j < len(row_q) else None
            tote = _coerce_int(row_t[j]) if j < len(row_t) else None
            if item is None or qty is None or tote is None or qty <= 0:
                continue
            tote_bins.setdefault(tote, []).extend([i + 1] * qty)
            tote_items.setdefault(tote, []).extend([item] * qty)

    blocks = {}
    for tote, bins in tote_bins.items():
        b = sorted(bins)
        blocks[tote] = {
            "first_bin": b[0],
            "last_bin": b[-1],
            "units": len(b),
            "internal_switches": sum(1 for k in range(1, len(b)) if b[k] != b[k - 1]),
            "items": tote_items[tote],
            "item_pairs": list(zip(bins, tote_items[tote])),
        }
    return blocks


def build_order_tote_map(blocks):
    """Map each order_id to the set of totes containing its items."""
    order_tote_map = {}
    for tote, block in blocks.items():
        for order_id, _ in block["item_pairs"]:
            order_tote_map.setdefault(order_id, set()).add(tote)
    return order_tote_map


def transition_cost(prev_tote, prev_last_bin, tote, blocks):
    if prev_tote is None:
        return 0.0
    c = TOTE_SWITCH_TIME
    if prev_last_bin != blocks[tote]["first_bin"]:
        c += BIN_SWITCH_TIME
    return c


def block_cost(tote, blocks):
    b = blocks[tote]
    return b["units"] * PLACE_TIME + b["internal_switches"] * BIN_SWITCH_TIME


def build_optionality_terms(blocks):
    totes = sorted(blocks.keys())
    first_bins = {t: blocks[t]["first_bin"] for t in totes}
    last_bins = {t: blocks[t]["last_bin"] for t in totes}

    compat = set()
    branch = {t: 0 for t in totes}
    for i in totes:
        for j in totes:
            if i == j:
                continue
            if last_bins[i] == first_bins[j]:
                compat.add((i, j))
                branch[i] += 1

    bin_freq = {}
    for t in totes:
        b = first_bins[t]
        bin_freq[b] = bin_freq.get(b, 0) + 1

    node_coeff = {}
    for t in totes:
        rarity = 1.0 / bin_freq[first_bins[t]]
        node_coeff[t] = OPT_BRANCH_WEIGHT * branch[t] - OPT_RARE_WEIGHT * rarity

    return {"compat": compat, "node_coeff": node_coeff}


def build_item_optionality_terms(item_pairs):
    n = len(item_pairs)
    order_ids = [p[0] for p in item_pairs]

    compat = set()
    branch = [0] * n
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if order_ids[i] == order_ids[j]:
                compat.add((i, j))
                branch[i] += 1

    freq = {}
    for oid in order_ids:
        freq[oid] = freq.get(oid, 0) + 1

    node_coeff = {}
    for i, (oid, _) in enumerate(item_pairs):
        node_coeff[i] = OPT_BRANCH_WEIGHT * branch[i] - OPT_RARE_WEIGHT / freq[oid]

    return {"compat": compat, "node_coeff": node_coeff}


def sequence_metrics(seq, blocks, terms, order_tote_map):
    """Evaluate a tote sequence. Returns total_time, sum_order_completion, optionality, objective."""
    n = len(seq)
    cumulative_time = 0.0
    prev_tote = None
    prev_last_bin = None
    tote_finish = {}

    for tote in seq:
        if prev_tote is not None:
            cumulative_time += transition_cost(prev_tote, prev_last_bin, tote, blocks)
        cumulative_time += block_cost(tote, blocks)
        tote_finish[tote] = cumulative_time
        prev_tote = tote
        prev_last_bin = blocks[tote]["last_bin"]

    total_time = cumulative_time

    # Order completion time = finish time of each order's last tote in sequence
    seq_pos = {tote: pos for pos, tote in enumerate(seq)}
    sum_completion = 0.0
    for order_id, totes in order_tote_map.items():
        in_seq = [t for t in totes if t in seq_pos]
        if in_seq:
            last_tote = max(in_seq, key=lambda t: seq_pos[t])
            sum_completion += tote_finish[last_tote]

    # Optionality (unchanged)
    optionality = 0.0
    prev = None
    for pos, tote in enumerate(seq, start=1):
        optionality += terms["node_coeff"][tote] * (n + 1 - pos)
        if prev is not None and (prev, tote) in terms["compat"]:
            optionality += OPT_EDGE_WEIGHT
        prev = tote

    objective = sum_completion - OPTIONALITY_LAMBDA * optionality
    return total_time, sum_completion, optionality, objective


def greedy_sequence(blocks, terms, order_tote_map):
    remaining = set(blocks.keys())
    seq = []
    prev_tote = None
    prev_last_bin = None
    cumulative_time = 0.0
    scheduled = set()
    n = len(remaining)

    while remaining:
        pos = len(seq) + 1
        best = None
        best_key = None
        for tote in sorted(remaining):
            trans = transition_cost(prev_tote, prev_last_bin, tote, blocks) if prev_tote else 0.0
            ft = cumulative_time + trans + block_cost(tote, blocks)

            new_scheduled = scheduled | {tote}
            # Time component: sum of ft for orders that newly complete at this tote
            time_inc = sum(
                ft for order_id, totes in order_tote_map.items()
                if totes <= new_scheduled and not totes <= scheduled
            )

            # Optionality (unchanged)
            opt_inc = terms["node_coeff"][tote] * (n + 1 - pos)
            if prev_tote is not None and (prev_tote, tote) in terms["compat"]:
                opt_inc += OPT_EDGE_WEIGHT

            obj_inc = time_inc - OPTIONALITY_LAMBDA * opt_inc
            key = (obj_inc, tote)
            if best_key is None or key < best_key:
                best_key = key
                best = tote

        trans = transition_cost(prev_tote, prev_last_bin, best, blocks) if prev_tote else 0.0
        cumulative_time += trans + block_cost(best, blocks)
        scheduled.add(best)
        seq.append(best)
        prev_tote = best
        prev_last_bin = blocks[best]["last_bin"]
        remaining.remove(best)

    return seq


def sequence_items_greedy(item_pairs, item_terms, completing_orders):
    """Greedy item sequencing. Time component uses completion times for completing_orders."""
    n = len(item_pairs)
    if n <= 1:
        return item_pairs[:]

    order_items = {}
    for idx, (oid, _) in enumerate(item_pairs):
        order_items.setdefault(oid, set()).add(idx)

    completing_set = set(completing_orders)
    remaining = set(range(n))
    seq = []
    cumulative_time = 0.0
    scheduled = set()
    completed = set()
    prev_idx = None
    prev_oid = None

    while remaining:
        pos = len(seq) + 1
        best = None
        best_key = None
        for idx in sorted(remaining):
            oid = item_pairs[idx][0]
            trans = BIN_SWITCH_TIME if (prev_oid is not None and prev_oid != oid) else 0.0
            local_time = cumulative_time + PLACE_TIME + trans

            new_scheduled = scheduled | {idx}
            # Time component: sum of local_time for completing orders that newly finish
            time_inc = sum(
                local_time for co in completing_set - completed
                if order_items.get(co, set()) <= new_scheduled
            )

            # Optionality (unchanged)
            opt_inc = item_terms["node_coeff"][idx] * (n + 1 - pos)
            if prev_idx is not None and (prev_idx, idx) in item_terms["compat"]:
                opt_inc += OPT_EDGE_WEIGHT

            obj_inc = time_inc - OPTIONALITY_LAMBDA * opt_inc
            key = (obj_inc, idx)
            if best_key is None or key < best_key:
                best_key = key
                best = idx

        oid = item_pairs[best][0]
        trans = BIN_SWITCH_TIME if (prev_oid is not None and prev_oid != oid) else 0.0
        cumulative_time += PLACE_TIME + trans
        scheduled.add(best)

        for co in completing_set - completed:
            if order_items.get(co, set()) <= scheduled:
                completed.add(co)

        seq.append(best)
        prev_idx = best
        prev_oid = oid
        remaining.remove(best)

    return [item_pairs[i] for i in seq]


def write_sorter(seq, blocks, out_path):
    cols = {0: "circle", 1: "pentagon", 2: "trapezoid", 3: "triangle", 4: "star", 5: "moon", 6: "heart", 7: "cross"}
    rows = {}
    for tote in seq:
        conv = ((blocks[tote]["first_bin"] - 1) % NUM_CONVEYORS) + 1
        rows.setdefault(conv, {name: 0 for name in cols.values()})
        for shape in blocks[tote]["items"]:
            if shape in cols:
                rows[conv][cols[shape]] += 1

    out = []
    for conv in sorted(rows):
        r = {"conv_num": conv}
        r.update(rows[conv])
        out.append(r)

    with out_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["conv_num", "circle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"])
        w.writeheader()
        for r in out:
            w.writerow(r)


def write_item_offload(seq, blocks, out_path):
    rows = []
    pos = 0
    for tote in seq:
        for item_type in blocks[tote]["items"]:
            pos += 1
            rows.append({"sequence_pos": pos, "item_type": item_type})

    with out_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sequence_pos", "item_type"])
        w.writeheader()
        w.writerows(rows)


all_bases = _get_input_bases(RUN_ID)
if not all_bases:
    raise RuntimeError("No input runs found. Ensure inputs/runs exists or set RUN_ID appropriately.")

aggregate = []
for base in all_bases:
    INPUT_ITEMTYPES = base / "order_itemtypes.csv"
    INPUT_QUANTITIES = base / "order_quantities.csv"
    INPUT_TOTES = base / "orders_totes.csv"

    blocks = build_blocks()
    order_tote_map = build_order_tote_map(blocks)
    terms = build_optionality_terms(blocks)
    seq = greedy_sequence(blocks, terms, order_tote_map)

    # Determine which orders complete at each tote (last tote in seq for that order)
    seq_pos = {tote: pos for pos, tote in enumerate(seq)}
    orders_completing_in_tote = {tote: [] for tote in blocks}
    for order_id, totes in order_tote_map.items():
        in_seq = [t for t in totes if t in seq_pos]
        if in_seq:
            last_tote = max(in_seq, key=lambda t: seq_pos[t])
            orders_completing_in_tote[last_tote].append(order_id)

    for tote in blocks:
        item_terms = build_item_optionality_terms(blocks[tote]["item_pairs"])
        completing = orders_completing_in_tote[tote]
        blocks[tote]["items"] = [p[1] for p in sequence_items_greedy(
            blocks[tote]["item_pairs"], item_terms, completing
        )]

    total_time, sum_completion, optionality, objective = sequence_metrics(seq, blocks, terms, order_tote_map)

    if RUN_ID == "all":
        run_out = OUT / "baseline_greedy_runs" / base.name
    elif RUN_ID is None:
        run_out = OUT
    else:
        run_out = OUT / "baseline_greedy_runs" / base.name
    run_out.mkdir(parents=True, exist_ok=True)

    with (run_out / "baseline_greedy_tote_sequence.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sequence_pos", "tote"])
        w.writeheader()
        for i, tote in enumerate(seq, start=1):
            w.writerow({"sequence_pos": i, "tote": tote})

    write_sorter(seq, blocks, run_out / "optimized_input_from_baseline_greedy.csv")
    write_item_offload(seq, blocks, run_out / "baseline_greedy_tote_item_plan.csv")

    row = {
        "run_name": base.name,
        "total_time": total_time,
        "sum_order_completion_time": sum_completion,
        "optionality_score": optionality,
        "objective_score": objective,
        "n_totes": len(seq),
        "n_units": sum(b["units"] for b in blocks.values()),
    }
    with (run_out / "baseline_greedy_summary.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        w.writeheader()
        w.writerow(row)
    aggregate.append(row)

if RUN_ID == "all":
    with (OUT / "baseline_greedy_all_runs_summary.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(aggregate[0].keys()))
        w.writeheader()
        w.writerows(aggregate)
    print(f"Processed {len(aggregate)} runs.")
    print("Wrote aggregate: outputs/baseline_greedy_all_runs_summary.csv")
else:
    print(f"Greedy objective score: {aggregate[0]['objective_score']:.3f}")
    print(f"Sum order completion time: {aggregate[0]['sum_order_completion_time']:.3f}")
    print("Wrote run outputs under outputs/")

Processed 50 runs.
Wrote aggregate: outputs/baseline_greedy_all_runs_summary.csv
